### 2. Generate a Timestamp Request

- data data.txt: The file to timestamp.
- sha256: Hash algorithm (match JAdES requirements).
- out request.tsq: Outputs the request.

In [20]:
%%bash

pwd
echo "example data" > data.txt
ls -l data.txt
openssl ts -query -data data.txt -sha256 -out request.tsq
ls -l request.tsq


/Users/ehaas/Documents/FHIR/USCDI5-Sandbox


-rw-r--r--  1 ehaas  staff  13 Sep 24 16:58 data.txt


Using configuration from /opt/homebrew/etc/openssl@3/openssl.cnf


-rw-r--r--  1 ehaas  staff  66 Sep 24 16:58 request.tsq


### 3. Create a Self-Signed Timestamp Response

- queryfile request.tsq: The timestamp request.
-  inkey tsakey.pem: Your private key.
- signer tsacert.pem: Your self-signed certificate.
- out response.tsr: Outputs the self-signed timestamp response.
  
Note: This bypasses the TSA’s secure time source; the time is from your system clock.

In [21]:
%%bash

dir='/Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert'

cat << EOF| tee "tsa_config.cnf"  # Write to tsa_config.cnf
# ===========Configuration for TSA timestamp ===========
# =================== update configuration manually with your values =======================
[ tsa ]
default_tsa = tsa_config

[ tsa_config ]
serial = $dir/serial
crypto_device = builtin
signer_cert = $dir/cert.pem
certs = $dir/cert.pem
signer_key = $dir/private-key.pem
signer_digest = sha256
default_policy = 1.2.3.4.1
other_policies = 1.2.3.4.2, 1.2.3.4.3
digests = sha256
accuracy = secs:1, millisecs:500, microsecs:100
ordering = yes
tsa_name = yes
ess_cert_id_chain = no
ess_cert_id_alg = sha256

# EOF
# don't edit the previous line


# ===========Configuration for TSA timestamp ===========
# =================== update configuration manually with your values =======================
[ tsa ]
default_tsa = tsa_config

[ tsa_config ]
serial = /Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert/serial
crypto_device = builtin
signer_cert = /Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert/cert.pem
certs = /Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert/cert.pem
signer_key = /Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert/private-key.pem
signer_digest = sha256
default_policy = 1.2.3.4.1
other_policies = 1.2.3.4.2, 1.2.3.4.3
digests = sha256
accuracy = secs:1, millisecs:500, microsecs:100
ordering = yes
tsa_name = yes
ess_cert_id_chain = no
ess_cert_id_alg = sha256

# EOF
# don't edit the previous line


In [22]:
%%bash


# ls -l /Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert/cert.pem
# ls -l /Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert/private-key.pem
# ls -l /Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert/serial
# ls -l request.tsq
# openssl x509 -in /Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert/cert.pem -text -noout

INKEY='/Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert/private-key.pem'
CERT='/Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert/cert.pem'
CONFIG='tsa_config.cnf'

# cat $CERT


openssl ts -reply -config $CONFIG -queryfile request.tsq -inkey $INKEY -signer $CERT -out response.tsr

Using configuration from tsa_config.cnf
Response has been generated.


In [23]:
%%bash 
CONFIG='tsa_config.cnf'
RESPONSE='response.tsr'
REQUEST='request.tsq'
CERT='/Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert/cert.pem'

ls -l $REQUEST
ls -l $RESPONSE
ls -l $CERT
ls -l $CONFIG


openssl ts -verify  -config $CONFIG  -in "$RESPONSE" -queryfile "$REQUEST"  -partial_chain -untrusted "$CERT"

openssl asn1parse -in $RESPONSE -inform der. > inspect.txt

## Self-signed with a chain not working

-rw-r--r--  1 ehaas  staff  66 Sep 24 16:58 request.tsq
-rw-r--r--  1 ehaas  staff  1151 Sep 24 16:58 response.tsr
-rw-r--r--  1 ehaas  staff  1996 Sep 24 15:15 /Users/ehaas/Documents/FHIR/davinci-ecdx/CDEX-Signatures/example_org_cert/cert.pem
-rw-r--r--  1 ehaas  staff  873 Sep 24 16:58 tsa_config.cnf


Using configuration from tsa_config.cnf
8045BB0401000000:error:17800064:time stamp routines:ts_verify_cert:certificate verify error:crypto/ts/ts_rsp_verify.c:192:Verify error:self-signed certificate


Verification: FAILED
